# 職員グループ別 離職シミュレーション（事務・大学教員・中高教員／Colab版）

**使い方（3ステップ）**
1. 記入済み様式3枚（`roster_form_jimu.csv` / `roster_form_daigaku.csv` /
   `roster_form_chuko.csv`）を Drive の `ColabNotebooks/employee-attrition/workplace/` に置く
2. 下の設定セル（グループ定義・定年など）を確認
3. 「ランタイム → すべてのセルを実行」

**出力**：①グループ別の翌年度離職確率（雇用形態内訳つき）
②グループ別の従業員数指数（現在=100）の推移比較 ③ベアゼロ vs 通常昇給の比較

一部の様式が未提出でも、あるものだけで動きます。マクロ情報は自動付与。

In [ ]:
# ── 設定 ─────────────────────────────────────────────────────────────
DATA_MODE  = 'gdrive'      # 'gdrive' / 'local' / 'synthetic'（動作確認）
GDRIVE_DATA_DIR = '/content/drive/MyDrive/ColabNotebooks/employee-attrition/data'
LOCAL_DATA_DIR  = '/path/to/JPSED/data'
WORKPLACE_DIR   = '/content/drive/MyDrive/ColabNotebooks/employee-attrition/workplace'
REPO_URL   = 'https://github.com/dsk-yshkw/employee-attrition'

# グループ定義：様式ファイル / JPSED職種コード / 定年
#   事務職員: 70=その他一般事務系職
#   大学教員: JPSEDに専用コードが無いため None（職種を欠測にして業種・年齢・勤続・
#             雇用形態で予測）。感度チェックには 220 を入れて再実行
#   中高教員: 220=教員（小中高）
GROUPS = {
    '事務職員': dict(roster='roster_form_jimu.csv',    occupation=70,   retirement_age=65),
    '大学教員': dict(roster='roster_form_daigaku.csv', occupation=None, retirement_age=68),
    '中高教員': dict(roster='roster_form_chuko.csv',   occupation=220,  retirement_age=62),
}

BASE_YEAR = 2025; YEARS = 5; N_SIMS = 300
INDUSTRY_CODE = 55            # 教育（大学・中高等）
WAGE_MODE = 'fixed'           # 'fixed'=ベアゼロ（定昇のみ） / 'model'=モデル賃金
TEISHO_RATE = 0.017           # 定昇率
FUTURE_INFLATION_PCT = 2.0    # 将来インフレ率想定（%）

In [ ]:
# ── 環境準備 ──────────────────────────────────────────────────────────
import subprocess, sys, os
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('employee-attrition'):
        subprocess.run(['git','clone',REPO_URL], check=True)
    os.chdir('employee-attrition')
    subprocess.run([sys.executable,'-m','pip','install','-q','scikit-learn>=1.4','pandas','numpy','matplotlib'], check=True)
    if DATA_MODE=='gdrive':
        from google.colab import drive; drive.mount('/content/drive')
if os.path.basename(os.getcwd()) in ('notebooks','workplace'): os.chdir('..')
sys.path.insert(0, os.getcwd())
SYNTH = (DATA_MODE=='synthetic')
DATA_DIR = 'data/synthetic' if SYNTH else (GDRIVE_DATA_DIR if DATA_MODE=='gdrive' else LOCAL_DATA_DIR)
if SYNTH:
    WORKPLACE_DIR='workplace'
    for g in GROUPS: GROUPS[g]['roster']='roster_form_example.csv'
print('DATA_DIR:', DATA_DIR)

## 1. JPSED で較正済みモデルを学習（数分）

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src.data.loader import DataLoader
from src.data.panel_builder import PanelBuilder
from src.data.transitions import TransitionFrameBuilder
from src.data.macro import MacroData
from src.features.assembler import FeatureAssembler
from src.models.transitions import TransitionModels

pb=PanelBuilder(DataLoader(DATA_DIR, synthetic=SYNTH)); fa=FeatureAssembler(); mac=MacroData()
tf, FEATS = TransitionFrameBuilder(pb, fa).build()
tm = TransitionModels(FEATS).fit(tf)
print('学習完了: %d 遷移 / 全国離職率 %.3f' % (len(tf), tf['separated'].mean()))

# 将来CPI経路（実績＋想定インフレで延長）
cpi_hist=mac.lookup()['cpi_all_items']; CPI={}; last=float(cpi_hist.iloc[-1]); ly=int(cpi_hist.index.max())
for y in range(int(cpi_hist.index.min()), BASE_YEAR+YEARS+1):
    if y in cpi_hist.index: CPI[y]=float(cpi_hist.loc[y]); last=CPI[y]
    elif y>ly: last*= (1+FUTURE_INFLATION_PCT/100); CPI[y]=last
def infl(y): return (CPI[y]/CPI[y-1]-1)*100
ref=tf.copy(); ref['age_band_lo']=(ref['age']//5*5)

## 2. 様式の読み込み → グループ別コホート

In [ ]:
EMP_MAP={'正規':1,'契約':4,'パート':2,'派遣':3,'嘱託':5}
SEX_MAP={'男':1,'女':2}

def cell_median(emp,age_lo,col):
    m=ref[(ref['contract_type']==emp)&(ref['age_band_lo']==age_lo)][col].median()
    if pd.isna(m): m=ref[ref['contract_type']==emp][col].median()
    if pd.isna(m): m=ref[col].median()
    return m

def build_cohort(roster_path, occupation, seed=12345):
    form=pd.read_csv(roster_path, encoding='utf-8-sig')
    form.columns=['emp','sex','age_band','n','avg_tenure','avg_salary']
    form['n']=pd.to_numeric(form['n'],errors='coerce').fillna(0).astype(int)
    form=form[form['n']>0].copy()
    if len(form)==0: return None
    form['age_lo']=form['age_band'].str.split('-').str[0].astype(float)
    form['age_hi']=form['age_band'].str.split('-').str[1].astype(float)
    rng=np.random.default_rng(seed); rows=[]
    for _,r in form.iterrows():
        emp=EMP_MAP[r['emp'].strip()]
        ten=r['avg_tenure'] if pd.notna(r['avg_tenure']) else cell_median(emp,r['age_lo'],'tenure_years')
        sal=r['avg_salary'] if pd.notna(r['avg_salary']) else cell_median(emp,r['age_lo'],'annual_income')
        for _ in range(int(r['n'])):
            age=rng.uniform(r['age_lo'], r['age_hi']+1)
            rows.append(dict(gender=SEX_MAP.get(r['sex'].strip(),np.nan), age=age,
                education=np.nan, has_spouse=np.nan, has_child=np.nan,
                num_children=np.nan, youngest_child_age=np.nan,
                contract_type=emp, industry=INDUSTRY_CODE, firm_size=np.nan,
                occupation=(np.nan if occupation is None else occupation),
                position=np.nan, weekly_hours=np.nan, is_regular=int(emp==1),
                tenure_years=float(np.clip(rng.normal(ten,2),0,max(age-18,0))),
                annual_income=float(max(sal*rng.uniform(0.85,1.15),50)),
                prev_annual_income=np.nan, salary_growth_rate=np.nan))
    c=pd.DataFrame(rows); r0=BASE_YEAR-1
    c['cpi']=CPI[r0]; c['inflation_rate']=infl(r0)
    c['real_annual_income']=c['annual_income']/(CPI[r0]/100)
    c['real_prev_annual_income']=np.nan; c['real_salary_growth_rate']=np.nan
    return c

cohorts={}
for g,cfg in GROUPS.items():
    path=os.path.join(WORKPLACE_DIR, cfg['roster'])
    if not os.path.exists(path):
        print(f'【スキップ】{g}: 様式が見つかりません ({path})'); continue
    c=build_cohort(path, cfg['occupation'])
    if c is None: print(f'【スキップ】{g}: 人数の記入がありません'); continue
    cohorts[g]=c; print(f'{g}: {len(c)} 人')

## 3. グループ別・翌年度の離職確率

In [ ]:
inv_emp={v:k for k,v in EMP_MAP.items()}
rows=[]
for g,c in cohorts.items():
    p=tm.p_separate(c[FEATS]); c['p_sep']=p
    rows.append(dict(グループ=g, 区分='合計', 人数=len(c),
                     平均離職確率=p.mean(), 期待離職者数=p.sum()))
    for k,sub in c.groupby('contract_type'):
        rows.append(dict(グループ=g, 区分=inv_emp[int(k)], 人数=len(sub),
                         平均離職確率=sub['p_sep'].mean(), 期待離職者数=sub['p_sep'].sum()))
tab=pd.DataFrame(rows).set_index(['グループ','区分']).round(3)
display(tab)

## 4. 数年シミュレーション（グループ別）と比較

各年：定年退出（グループ別の定年、確定的）→ 中途離職の抽選 → 残留者更新
（年齢+1・勤続+1・賃金ルール・実質化）。性別・家族は固定。

In [ ]:
def simulate_workplace(ci, retirement_age, years=YEARS, n_sims=N_SIMS,
                       wage_mode=WAGE_MODE, teisho=TEISHO_RATE, seed0=0):
    N0=len(ci); recs=[]
    for s in range(n_sims):
        rng=np.random.default_rng(seed0+s); st=ci.copy()
        for k in range(1, years+1):
            year=BASE_YEAR+k-1
            retire = st['age']>=retirement_age if retirement_age is not None else pd.Series(False,index=st.index)
            n_ret=int(retire.sum()); st=st[~retire]
            if len(st)>0:
                p=tm.p_separate(st[FEATS]); q=rng.random(len(st))<p
                n_quit=int(q.sum()); st=st[~q]
            else: n_quit=0
            if len(st)>0:
                old=st['annual_income'].to_numpy(float)
                new=old*(1+teisho) if wage_mode=='fixed' else tm.wage_stay(st[FEATS])
                cpi=CPI[year]
                st=st.assign(age=st['age']+1, tenure_years=st['tenure_years']+1,
                    prev_annual_income=old, annual_income=new,
                    salary_growth_rate=np.clip((new-old)/old,-1,3),
                    cpi=cpi, inflation_rate=infl(year),
                    real_prev_annual_income=st['real_annual_income'],
                    real_annual_income=new/(cpi/100))
                st=st.assign(real_salary_growth_rate=np.clip(
                    (st['real_annual_income']-st['real_prev_annual_income'])
                    /st['real_prev_annual_income'],-1,3))
            recs.append(dict(sim=s,k=k,year=year+1,remain=len(st),quits=n_quit,retirements=n_ret))
    return pd.DataFrame(recs), N0

results={}
for g,c in cohorts.items():
    r,N0=simulate_workplace(c.drop(columns=['p_sep']), GROUPS[g]['retirement_age'])
    results[g]=(r,N0)
    gk=r.groupby('k')
    summ=pd.DataFrame({'暦年':gk['year'].first(),
        '指数（現在=100）':(gk['remain'].mean()/N0*100).round(1),
        '5%点':(gk['remain'].quantile(.05)/N0*100).round(1),
        '95%点':(gk['remain'].quantile(.95)/N0*100).round(1),
        '中途離職/年':gk['quits'].mean().round(2),
        '定年退職/年':gk['retirements'].mean().round(2)})
    print(f"\n■ {g}（{N0}人, 定年{GROUPS[g]['retirement_age']}歳）")
    display(summ)

In [ ]:
COLORS={'事務職員':'#0072B2','大学教員':'#D55E00','中高教員':'#009E73'}
fig,ax=plt.subplots(1,2,figsize=(12,4.5))
for g,(r,N0) in results.items():
    gk=r.groupby('k'); ks=np.array(sorted(r['k'].unique()))
    m=gk['remain'].mean()/N0*100
    ax[0].plot(np.r_[0,ks], np.r_[100,m], 'o-', color=COLORS.get(g,'#888'), label=g)
    ax[0].fill_between(ks, gk['remain'].quantile(.05)/N0*100,
                       gk['remain'].quantile(.95)/N0*100, alpha=.12, color=COLORS.get(g,'#888'))
    tot=gk['quits'].mean()+gk['retirements'].mean()
    ax[1].plot(ks, tot/N0*100, 'o-', color=COLORS.get(g,'#888'), label=g)
ax[0].set_xlabel('years ahead'); ax[0].set_ylabel('headcount index (now=100)')
ax[0].set_title('Workforce index by group (90% band)'); ax[0].legend()
ax[1].set_xlabel('years ahead'); ax[1].set_ylabel('leavers / initial headcount (%)')
ax[1].set_title('Annual leaver rate (quits + retirement)'); ax[1].legend()
plt.tight_layout(); plt.show()

## 5. シナリオ比較：ベアゼロ vs 通常昇給（グループ別）

In [ ]:
fig,axes=plt.subplots(1,len(cohorts),figsize=(4.2*len(cohorts),3.8),squeeze=False)
tabrows=[]
for j,(g,c) in enumerate(cohorts.items()):
    ax=axes[0][j]
    for mode,label,color in [('fixed','ベアゼロ','#D55E00'),('model','通常昇給','#0072B2')]:
        r,N0=simulate_workplace(c.drop(columns=['p_sep']), GROUPS[g]['retirement_age'], wage_mode=mode)
        m=r.groupby('k')['remain'].mean()/N0*100
        ax.plot(np.r_[0,m.index], np.r_[100,m], 'o-', color=color, label=label)
        tabrows.append(dict(グループ=g, シナリオ=label, **{f'{YEARS}年後指数': round(m.iloc[-1],1)}))
    ax.set_title(g); ax.set_xlabel('years ahead'); ax.set_ylabel('index (now=100)'); ax.legend()
plt.tight_layout(); plt.show()
display(pd.DataFrame(tabrows).pivot(index='グループ', columns='シナリオ'))

## 読み方の注意

- **大学教員は JPSED に専用職種コードが無い**ため、職種を欠測として業種・年齢・
  勤続・年収・**雇用形態**で予測している（特任・任期付は「契約」、非常勤講師は
  「パート」として様式に記入することが重要）。テニュア制度など大学固有の
  メカニズムは守備範囲外なので、他グループより幅を持って読むこと。
  感度チェック：GROUPS の occupation を 220 に変えて再実行し結果の幅を見る
- 中高教員（コード220）は JPSED n≈3,000 の裏付けあり（正規5.3%・契約18.5%など）
- モデルは全国パネルで学習。職場固有の事情は観測変数を通じてのみ反映
- 定年は年齢ルールで確定処理。補充採用は含まない（純減の推移）